In [1]:
import numpy as np
import optuna
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

train = np.loadtxt("Datasets/pendigits.tra", delimiter=",")
test = np.loadtxt("Datasets/pendigits.tes", delimiter=",")
Xtr, ytr = train[:, :16] / 100, train[:, 16].astype(int)
Xte, yte = test[:, :16] / 100, test[:, 16].astype(int)

Xa, Xval, ya, yval = train_test_split(Xtr, ytr, test_size=0.2, stratify=ytr, random_state=42)


def train_net(X, y, width, lr):
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(16, width), nn.ReLU(), nn.Linear(width, 32), nn.ReLU(), nn.Linear(32, 10))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)
    for epoch in range(10):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 128):
            idx = perm[i:i + 128]
            loss = loss_fn(model(X[idx]), y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


def accuracy(model, X, y):
    with torch.no_grad():
        pred = model(torch.tensor(X, dtype=torch.float32)).argmax(dim=1).numpy()
    return float((pred == y).mean())


def objective(trial):
    width = trial.suggest_categorical("width", [32, 64, 128])
    lr = trial.suggest_float("lr", 0.0005, 0.01, log=True)
    model = train_net(Xa, ya, width, lr)
    return accuracy(model, Xval, yval)


sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=20)

print("Best width:", study.best_params["width"])
print("Best learning rate:", round(study.best_params["lr"], 5))
print("Best validation accuracy:", round(study.best_value, 4))

best = train_net(Xtr, ytr, study.best_params["width"], study.best_params["lr"])
print("Test accuracy with best settings:", round(accuracy(best, Xte, yte), 4))


[I 2026-11-25 15:02:09,746] A new study created in memory with name: no-name-bb2ba8b1-d61d-4d65-897b-38c3ed72d75c
[I 2026-11-25 15:02:12,699] Trial 0 finished with value: 0.9659773182121414 and parameters: {'width': 64, 'lr': 0.0030049873591901574}. Best is trial 0 with value: 0.9659773182121414.
[I 2026-11-25 15:02:13,062] Trial 1 finished with value: 0.9586390927284857 and parameters: {'width': 32, 'lr': 0.0066971673533752435}. Best is trial 0 with value: 0.9659773182121414.
[I 2026-11-25 15:02:13,446] Trial 2 finished with value: 0.9846564376250834 and parameters: {'width': 64, 'lr': 0.009138013915892867}. Best is trial 2 with value: 0.9846564376250834.
[I 2026-11-25 15:02:13,813] Trial 3 finished with value: 0.8485657104736491 and parameters: {'width': 32, 'lr': 0.0008661333735273134}. Best is trial 2 with value: 0.9846564376250834.
[I 2026-11-25 15:02:14,195] Trial 4 finished with value: 0.923949299533022 and parameters: {'width': 64, 'lr': 0.0011963764382790319}. Best is trial 2 

Best width: 64
Best learning rate: 0.00914
Best validation accuracy: 0.9847
Test accuracy with best settings: 0.9443
